# 00. Dataset Construction and Storage — GSE287331
This notebook builds the methylation matrix for the tumor-proximity breast cancer cohort, unifying tumor, case-benign (AN, OQ, CUB), and healthy donated breast (HDB) samples into a single transposed, labeled Parquet table for downstream exploratory and comparative analyses.

**Source: GEO accession GSE287331, platform Illumina Infinium MethylationEPIC v1.0 BeadChip**


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-preparation-GSE287331                   ║
# ║ Description:  Construction of the core methylation matrix        ║
# ║               (GSE287331) — Parquet storage                      ║
# ║ Dataset(s):   GSE287331                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 09-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
import os, gzip, json, gc
from pathlib import Path
from typing import List, Optional
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


## 1. Read TXT → write Parquet (LZ4)

In [ ]:
# BUILD ONE PARQUET FILE, TRANSPOSED
INPUT_PATH   = "/kaggle/input/gse287331-betas-csv/GSE287331_betas_processed.csv"   
OUTPUT_FILE  = "/kaggle/working/GSE287331_lz4.parquet"
TMP_MEMMAP   = "/path/to/tmp/matrix_memmap.f32"   # deleted at the end if KEEP_MEMMAP=False
ID_COL       = "id_sample"
PROBE_COL    = None     # if None -> first column is assumed CpG/probe id
CPG_BLOCK    = 50_000   # rows per chunk (CpGs per block)
FLOAT_DTYPE  = np.float32
NA_STRINGS   = ["NA", "NaN", "nan", "", "null", "NULL"]
KEEP_MEMMAP  = False     # set True if you want to keep the memmap for debugging

def guess_sep(path: str) -> str:
    p = str(path).lower()
    if p.endswith(".tsv") or p.endswith(".txt"): return "\t"
    if p.endswith(".csv") or p.endswith(".csv.gz"): return ","
    # fallback sniff
    with open(path, "rb") as fh:
        first = fh.readline().decode("utf-8", "ignore")
    return "\t" if first.count("\t") >= first.count(",") else ","

def open_text(path: str):
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path, "rt", encoding="utf-8", newline="")

def count_cpg_rows(path: str) -> int:
    with open_text(path) as f:
        # skip header
        header = next(f)
        return sum(1 for _ in f)

def ensure_parent(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

# Discover schema
sep = guess_sep(INPUT_PATH)
hdr = pd.read_csv(INPUT_PATH, sep=sep, nrows=0)
cols = hdr.columns.tolist()
probe_col = PROBE_COL or cols[0]
sample_cols = [c for c in cols if c != probe_col]
num_samples = len(sample_cols)
num_cpgs = count_cpg_rows(INPUT_PATH)

print(f"[INFO] sep={repr(sep)} | probe_col={probe_col} | #samples={num_samples:,} | #CpGs={num_cpgs:,}")

# Allocate memmap (Fortran order so each column is contiguous)
#     shape = (samples, CpGs)
ensure_parent(TMP_MEMMAP)
mm = np.memmap(TMP_MEMMAP, dtype=FLOAT_DTYPE, mode="w+", shape=(num_samples, num_cpgs), order="F")

# We'll collect CpG IDs to use as column names
cpg_names: List[str] = []
written = 0

# 3) Fill memmap column-blocks by reading CpGs in chunks
reader = pd.read_csv(
    INPUT_PATH,
    sep=sep,
    dtype={probe_col: "string"},
    chunksize=CPG_BLOCK,
    na_values=NA_STRINGS,
    keep_default_na=True,
    low_memory=False,
)

for chunk in reader:
    # reorder and cast
    chunk = chunk[[probe_col] + sample_cols]
    block_cpgs = chunk[probe_col].astype("string").tolist()
    vals = chunk[sample_cols].to_numpy(dtype=FLOAT_DTYPE, copy=False).T  # shape: [samples, block]

    start = written
    end = start + vals.shape[1]
    mm[:, start:end] = vals

    cpg_names.extend(block_cpgs)
    written = end

    del chunk, vals
    gc.collect()

assert written == num_cpgs, f"Written {written} != expected {num_cpgs}"
print(f"[OK] Memmap filled: shape={mm.shape}, dtype={mm.dtype}, order=F")

# 4) Build Arrow Table column-by-column (zero-copy from memmap)
#     NOTE: because memmap is Fortran-order, mm[:, j] is contiguous
print("[INFO] Building Arrow arrays (this may take a while for ~866k columns)...")
arrays = [pa.array(sample_cols, type=pa.string())]
names  = [ID_COL]

# Create Arrow arrays without copying data
fa_type = pa.float32()
for j, cpg in enumerate(cpg_names):
    col = mm[:, j]                    # contiguous 1-D view (Fortran order)
    arr = pa.array(col, type=fa_type) # zero-copy when possible
    arrays.append(arr)
    names.append(str(cpg))

table = pa.Table.from_arrays(arrays, names=names)
print(f"[INFO] Arrow table: {table.shape} (rows, cols)")

# 5) Write ONE Parquet file (LZ4)
ensure_parent(OUTPUT_FILE)
try:
    pq.write_table(
        table,
        OUTPUT_FILE,
        compression="lz4",
        use_dictionary=False,               # faster & smaller for floats off
        data_page_size=1<<20,               # 1MB pages
        write_statistics=True,
    )
    print(f"[DONE] Wrote single Parquet -> {OUTPUT_FILE}")
finally:
    # Free big objects
    del table, arrays
    gc.collect()
    if not KEEP_MEMMAP:
        try:
            os.remove(TMP_MEMMAP)
        except Exception:
            pass
